# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第14章：分類とクラスタリング：判別分析・混合分布・二峰性


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


実データ: Peterson & Barney (1952)（phonToolsパッケージ収録）／参照: Hartigan & Hartigan (1985) dip検定／Scrucca et al. mclust。

In [ ]:
install.packages(c("MASS", "mclust", "diptest", "phonTools"), repos = "https://cloud.r-project.org")
library(MASS)      # 線形判別分析 lda()
library(mclust)    # 正規混合分布モデル
library(diptest)   # Hartiganのdip検定（単峰性からの逸脱を検定する）
library(phonTools) # Peterson & Barney (1952) の実測データ
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


■ 「どれくらい違うか」ではなく「区別できるか」を測る：

第3章から第13章までは、条件によって平均がどれだけ動くかを推定してきた。しかし音声研究の問いには、別の形のものもある。**「この2つの音響特徴だけで、母音カテゴリを言い当てられるか」****「この分布は1つの山か、それとも2つの集団が混ざっているか」**といった問いである。前者は分類、後者はクラスタリングの問題であり、回帰とは異なる道具を使う。

この章では実測データ——Peterson & Barney (1952) の母音フォルマント——を使い、分類がどれくらいうまくいくか、そして第2章で学んだ話者正規化が分類精度にどう効くかを確かめる。

■ 線形判別分析：フォルマントから母音を言い当てる：

In [ ]:
data(pb52)

pb <- pb52 %>%
  mutate(talker_type = recode(as.character(type), m = "男性", w = "女性", c = "子ども")) %>%
  group_by(speaker) %>%
  # 第2章と同じLobanov正規化（話者ごとのzスコア）。
  # ⚠️ 正規化のパラメータ（各話者の平均・SD）は「その話者の全トークン」
  #   （訓練・テストに分ける前の全体）から計算される。Lobanov・Nearey1は
  #   F1・F2の値だけを使い、母音カテゴリのラベルは使わない教師なしの変換
  #   なのでラベル漏洩ではないが、**テスト話者についても音響測定値そのものは
  #   正規化の計算に使われている**——「その話者の母音一式は既に観測できている」
  #   トランスダクティブな設定になる。1トークンだけの完全に未知な話者を
  #   分類する課題とは異なる、より緩やかな評価設定であることに注意する
  #   （詳細は本文14.2節）。ただし後述のWatt & Fabricius法は、基準となる
  #   母音（/i/・/ɑ/相当）を特定するのに母音ラベルそのものを使うため、
  #   この「ラベルは未知」という説明は当てはまらない点にも注意する。
  mutate(f1z = as.numeric(scale(f1)), f2z = as.numeric(scale(f2))) %>%
  ungroup()

# ★★訓練・テストは「トークン単位」ではなく「話者単位」で分ける。
#   トークン単位でランダムに分けると（例: sample(nrow(pb), ...)）、
#   同じ話者の他のトークンが訓練データに混ざり込む。その場合、モデルは
#   「未知の話者の母音を当てる」課題ではなく「既知の話者の、まだ見ていない
#   トークンを当てる」課題を解くことになり、精度は本来より高く出る。
set.seed(5)
speakers <- unique(pb$speaker)
train_speakers <- sample(speakers, floor(0.7 * length(speakers)))
train <- pb %>% filter(speaker %in% train_speakers)
test  <- pb %>% filter(!speaker %in% train_speakers)

accuracy <- function(model, data) mean(predict(model, data)$class == data$vowel)

lda_raw  <- lda(vowel ~ f1 + f2, data = train)     # 生のHz
lda_norm <- lda(vowel ~ f1z + f2z, data = train)   # 話者内で正規化

cat("10母音の判別・未知話者への汎化（当てずっぽうなら10%、この1回の分割のみ）\n")
cat("  生のHz F1,F2       :", round(accuracy(lda_raw, test), 3), "\n")
cat("  話者内正規化 F1,F2 :", round(accuracy(lda_norm, test), 3), "\n")

# どの母音どうしが混同されやすいか
pred <- predict(lda_norm, test)$class
confusion <- as.data.frame(table(true = test$vowel, pred = pred)) %>%
  filter(true != pred) %>% arrange(desc(Freq)) %>% head(5)
cat("\n最も混同された組み合わせ（この1回の分割）:\n")
print(confusion)

# --- 実行結果 ---
# 10母音の判別・未知話者への汎化（当てずっぽうなら10%、この1回の分割のみ）
#   生のHz F1,F2       : 0.713
#   話者内正規化 F1,F2 : 0.935
#
# 最も混同された組み合わせ（この1回の分割）:
#   true pred Freq
#      u    U    5
#      I    E    4
#      A    V    4
#      {    E    3
#      O    u    3
#
# 【出力の読み方】
# ・話者内正規化は、生のHzより判別精度を上げるはずである。使っている
#   情報はどちらもF1とF2の2つだけで、変えたのは
#   「話者ごとの声道長の違いを取り除いたかどうか」だけである。
# ・混同表が音声学的に筋の通った形をしているか確認する。
#   隣り合う母音どうしの混同（例: u と U）は自然だが、
#   もし i と A のように空間の端どうしが混同されていたら、
#   それは母音の性質ではなくコードの誤りを疑う場面である。
# ・⚠️ **ただし、この93.5%という数値を代表精度として報告してはならない。**
#   set.seed(5)という1つの分割がたまたま返した値にすぎない。次のセルで、
#   分割を100回繰り返して平均・SD・分布を確認する。


■ 話者単位の分割を100回繰り返す：

1回の分割の結果がどれだけ運に左右されるかを確かめるため、訓練・テストへの話者単位の分割を100回繰り返し、正規化・F3追加・3つの正規化手法・話者タイプ判別・2母音判別のすべてをまとめて評価する。

In [ ]:
# ⏳ 数十秒（LDAは軽いモデルなので、800回近くのfitでもすぐ終わる）。

# Nearey1（formant-intrinsic、対数平均を引く）
pb <- pb %>% group_by(speaker) %>%
  mutate(f1_nearey = log(f1) - mean(log(f1)), f2_nearey = log(f2) - mean(log(f2)),
         f3z = as.numeric(scale(f3))) %>%
  ungroup()

# Watt & Fabricius（3つの角の重心を基準点にする）
# ⚠️ d$vowel == "i" / "A" で母音ラベルを直接使って角を特定している。
#   Lobanov・Nearey1と違い、この手法はテスト話者についても母音ラベルの
#   一部を使う点で、より緩い評価設定になっている（本文14.2節参照）。
speakers <- unique(pb$speaker)
wf_list <- lapply(speakers, function(sp) {
  d <- pb %>% filter(speaker == sp)
  Fi1 <- mean(d$f1[d$vowel == "i"]); Fi2 <- mean(d$f2[d$vowel == "i"])
  Fa1 <- mean(d$f1[d$vowel == "A"]); Fa2 <- mean(d$f2[d$vowel == "A"])
  Fu1 <- Fi1; Fu2 <- Fi1                       # 計算上の/u/: F1(u')=F2(u')=F1(i)
  S1 <- (Fi1 + Fa1 + Fu1) / 3; S2 <- (Fi2 + Fa2 + Fu2) / 3
  d$f1_wf <- d$f1 / S1; d$f2_wf <- d$f2 / S2
  d
})
pb <- bind_rows(wf_list)

acc      <- function(m, d) mean(predict(m, d)$class == d$vowel)
acc_type <- function(m, d) mean(predict(m, d)$class == d$talker_type)

set.seed(100)
n_rep <- 100
res <- vector("list", n_rep)
for (i in 1:n_rep) {
  train_speakers <- sample(speakers, floor(0.7 * length(speakers)))
  train <- pb %>% filter(speaker %in% train_speakers)
  test  <- pb %>% filter(!speaker %in% train_speakers)

  m_raw       <- lda(vowel ~ f1 + f2, data = train)
  m_lob       <- lda(vowel ~ f1z + f2z, data = train)
  m_lob_f3raw <- lda(vowel ~ f1z + f2z + f3, data = train)
  m_lob_f3z   <- lda(vowel ~ f1z + f2z + f3z, data = train)
  m_nearey    <- lda(vowel ~ f1_nearey + f2_nearey, data = train)
  m_wf        <- lda(vowel ~ f1_wf + f2_wf, data = train)
  m_type_raw  <- lda(talker_type ~ f1 + f2, data = train)
  m_type_norm <- lda(talker_type ~ f1z + f2z, data = train)

  pb2 <- pb %>% filter(vowel %in% c("i", "I")) %>% mutate(vowel = droplevels(vowel))
  sp2 <- unique(pb2$speaker); tr2 <- sample(sp2, floor(0.7 * length(sp2)))
  train2 <- pb2 %>% filter(speaker %in% tr2); test2 <- pb2 %>% filter(!speaker %in% tr2)
  m_ii <- lda(vowel ~ f1z + f2z, data = train2)

  res[[i]] <- tibble(
    raw = acc(m_raw, test), lobanov = acc(m_lob, test),
    lobanov_f3raw = acc(m_lob_f3raw, test), lobanov_f3norm = acc(m_lob_f3z, test),
    nearey = acc(m_nearey, test), wf = acc(m_wf, test),
    type_raw = acc_type(m_type_raw, test), type_norm = acc_type(m_type_norm, test),
    ii = acc(m_ii, test2)
  )
}
allres <- bind_rows(res)

summary_table <- allres %>%
  summarise(across(everything(), list(
    mean = ~mean(.) * 100, sd = ~sd(.) * 100,
    p2.5 = ~quantile(., .025) * 100, p97.5 = ~quantile(., .975) * 100
  ))) %>%
  pivot_longer(everything(), names_to = c("指標", ".value"), names_pattern = "(.*)_(mean|sd|p2.5|p97.5)")
print(summary_table, n = 20)

# --- 実行結果 ---
# 指標             mean    sd  p2.5 p97.5
# raw              69.5   3.4  63.1  76.4
# lobanov          91.4   1.5  88.5  94.0
# lobanov_f3raw    91.7   1.5  88.7  94.5
# lobanov_f3norm   92.2   1.3  89.8  94.8
# nearey           89.4   1.7  86.2  92.2
# wf               84.2   2.0  80.8  87.5
# type_raw         54.0   4.6  43.0  60.2
# type_norm        39.5   8.3  21.7  52.2
# ii               99.4   0.7  97.8 100.0
#
# 【出力の読み方】
# ・**100回の分割にわたる平均は91.4%（SD 1.5pt）。** 前のセルで見た
#   set.seed(5)の93.5%は、平均より2標準偏差近く高い、やや都合のよい
#   結果だった。話者単位の分類精度は、都合のよい1回の分割ではなく、
#   このような複数回の分割にわたる平均・SD・分位点で報告する。
# ・**F3を足す効果は、正規化してから足すほうが一貫して良い**
#   （生のF3を足す91.7% < 正規化したF3を足す92.2%）。SDも
#   1.5→1.5→1.3ptとわずかに縮む。他の特徴量と同じ前処理を通す
#   ことの実例になっている。
# ・**Lobanov（91.4%）> Nearey1（89.4%）> Watt&Fabricius（84.2%）**
#   という順序が偶然かどうかは、各手法のSD（1.5〜2.0pt）どうしを比べる
#   のではなく、**同じ100回の分割内での対応差**（allres$lobanov -
#   allres$wf のように、同じ分割番号どうしの差）で判断する。
#   平均差は7.23ポイント・SD 1.60ポイント（2.5-97.5%タイル
#   [3.80, 10.57]）となり、対応差のSDが各手法のmarginal SDより
#   小さいのは、同じ分割の「易しさ・難しさ」が両手法に共通して
#   効くためである。★ここで`t.test(paired = TRUE)`のようなふつうの
#   対応ありt検定を当ててp値や信頼区間を計算してはいけない——
#   100回の分割は同じ話者データを繰り返し使い回しており、対応あり
#   t検定が前提とする独立性を満たさない（Dietterich 1998）。平均・
#   SD・分位点は記述統計として示すにとどめる。
# ・話者タイプの判別は、正規化すると54.0%→39.5%と予想どおり下がる
#   （SDが8.3ptと大きいのは、母音間の分散を消す操作が話者タイプの
#   判別を不安定にすることの表れでもある）。
# ・/i/ vs /I/ の2クラス判別は99.4%（SD 0.7pt）で、10母音（91.4%）
#   よりはっきり高い。分類の難しさはクラス数とクラス間の距離で決まる。


■ 次元を圧縮して眺める：主成分分析と多次元尺度構成法：

複数の音響指標を要約したいとき、あるいは類似度しか分からないときに使う道具。

In [ ]:
# 主成分分析: 話者内正規化したF1・F2・F3を2〜3個の合成変数に圧縮する
pc <- prcomp(pb[, c("f1z","f2z","f3z")], scale. = FALSE)
summary(pc)
round(pc$rotation, 3)  # 負荷量: 各主成分に元の変数がどれだけ寄与しているか

# 多次元尺度構成法: 母音カテゴリの重心間の距離だけから2次元空間を復元する
vowel_means <- pb %>% group_by(vowel) %>% summarise(f1z = mean(f1z), f2z = mean(f2z), .groups = "drop")
dist_mat <- dist(vowel_means[, c("f1z", "f2z")])
mds <- cmdscale(dist_mat, k = 2)
cat("MDS次元1 と 実測F2zの相関:", round(cor(mds[,1], vowel_means$f2z), 3), "\n")
cat("MDS次元2 と 実測F1zの相関:", round(cor(mds[,2], vowel_means$f1z), 3), "\n")

# --- 実行結果 ---
# Importance of components:
#                           PC1    PC2    PC3
# Standard deviation     1.2360 0.9248 0.6849
# Proportion of Variance 0.5356 0.2999 0.1645
# Cumulative Proportion  0.5356 0.8355 1.0000
#
# 負荷量:
#        PC1   PC2    PC3
# f1z  0.407 0.890  0.205
# f2z -0.669 0.138  0.730
# f3z -0.621 0.434 -0.652
#
# MDS次元1 と 実測F2zの相関: 0.834
# MDS次元2 と 実測F1zの相関: 0.652
#
# 【出力の読み方】
# ・第1主成分だけで分散の53.6%、第2主成分まで含めると83.6%を説明する。
#   3つの音響指標を、情報の8割以上を保ったまま2次元に圧縮できたことになる。
# ・主成分そのものは意味を持たない。負荷量を見て解釈する。第1主成分は
#   F2・F3と逆向きに強く相関しており、おおよそ「前舌性」に近い軸として読める。
# ・MDSが復元した2つの次元は、実測のF2z（相関0.834）・F1z（相関0.652）と
#   それぞれ対応するが、完全な一致ではない。**これは情報が失われたのではなく、
#   MDSの解が回転・反転に対して不定である**ためである。実際、MDSの配置に
#   最適な回転（プロクラステス回転）を1つ当てはめると、元のF1z・F2zの
#   重心配置と誤差10⁻¹⁵（浮動小数点誤差の範囲）で完全に一致する——MDSは
#   元の情報を100%保っているが、軸の向き自体は勝手に選ばれる。
#   **復元された次元に音声学的な意味を与える前に、既知の測定値との相関を
#   確認する**手順が実務上の鉄則になる。


■ 分布は1つの山か、2つの集団の混合か：

音声研究では「このデータは2つの下位集団に分かれているのではないか」という問いがしばしば立つ。有声破裂音のVOTが、前voicing（負のVOT）を使う発話と使わない発話に分かれる場合などである。分布が二峰かどうかは、ヒストグラムの見た目では判断できない。ビン幅を変えれば山は増えたり消えたりするからである。

In [ ]:
set.seed(9)
unimodal <- rnorm(300, mean = 50, sd = 10)                        # 1つの山
bimodal  <- c(rnorm(150, mean = 20, sd = 5), rnorm(150, 80, 8))   # 2つの山

# Hartiganのdip検定: 帰無仮説は「単峰である」
cat("単峰データ: dip検定 p =", round(dip.test(unimodal)$p.value, 4), "\n")
cat("二峰データ: dip検定 p =", round(dip.test(bimodal)$p.value, 4), "\n")

# 正規混合分布モデル: 成分がいくつあるかをBICで選ばせる
mc <- Mclust(bimodal, G = 1:3, verbose = FALSE)
cat("\nmclustが選んだ成分数:", mc$G, "\n")
cat("各成分の平均:", round(mc$parameters$mean, 2), "\n")
cat("各成分の割合:", round(mc$parameters$pro, 3), "\n")

# --- 実行結果 ---
# 単峰データ: dip検定 p = 0.9184
# 二峰データ: dip検定 p = 0
#
# mclustが選んだ成分数: 2
# 各成分の平均: 20.38 80.4
# 各成分の割合: 0.5 0.5
#
# 【出力の読み方】
# ・dip検定の帰無仮説は「分布は単峰である」なので、**p値が小さいほど
#   二峰（あるいはそれ以上）の証拠が強い**。単峰データではp = 0.9184で
#   単峰性を否定できず、二峰データではp = 0で明確に否定される。
#   期待どおりの動作である。
# ・★p値の向きを取り違えないこと。「p > 0.05 だから単峰である」とは言えない。
#   検出力が足りないだけかもしれない。言えるのは
#   「二峰だという証拠は得られなかった」までである。
# ・混合分布モデルは、より多くの情報を返す。成分数を2と判定し、
#   平均を20.38と80.4（生成時の20と80）、混合比を0.5対0.5
#   （生成時も0.5対0.5）と、いずれも正確に回収している。
# ・**dip検定と混合分布モデルは役割が違う。** dip検定は「単峰か否か」だけを
#   答える検定であり、混合分布モデルは「いくつの集団が、どこに、どの割合で
#   あるか」を推定する。まずdip検定で二峰性の証拠を確認し、
#   次に混合分布モデルで中身を記述する、という順序が使いやすい。
# ・★混合分布モデルの重大な注意点は、**成分数を増やせばどんな分布にも
#   当てはまってしまう**ことである。正規分布1つのデータに3成分を当てても、
#   当てはまり自体は良くなる。だからBICのような、複雑さに罰則を課す
#   基準で成分数を選ぶ。上では G = 1:3 の範囲でBICが最良のものを選ばせている。
# ・★さらに根本的な注意として、**統計的に2成分が選ばれたことは、
#   その2つが言語的に意味のあるカテゴリであることを意味しない。**
#   録音セッションの違い、マイクの違い、話者の下位集団など、
#   別の理由で分布が分かれることは多い。成分が何に対応するかは、
#   モデルではなくデータの素性を調べて確かめる必要がある。


■ 演習：

1. `pb52` で、話者タイプ（男性・女性・子ども）を F1・F2 から判別できるか試せ。正規化前と後で精度がどう変わるかを予想してから、上のセルで求めた`allres`（100回分の結果）を使って確認せよ。
2. 母音を /i/ と /I/ の2つだけに絞って判別し、10母音のときより精度が上がることを `allres` で確認せよ。
3. `pb52` の /u/ のF2について dip検定を行い、二峰性があるか調べよ。

■ 演習の解答：

In [ ]:
# 1・2. 前のセルで求めたallres（100回の分割の結果）をそのまま使う。
#       ★1回だけ分割し直して確認しない——それでは「都合のよい1回」を
#       選び直しているだけになり、前のセルで批判した問題を繰り返すことになる。
allres %>%
  summarise(
    talker_raw_mean  = mean(type_raw) * 100, talker_raw_sd  = sd(type_raw) * 100,
    talker_norm_mean = mean(type_norm) * 100, talker_norm_sd = sd(type_norm) * 100,
    ii_mean = mean(ii) * 100, ii_sd = sd(ii) * 100,
    vowel10_mean = mean(lobanov) * 100
  ) %>% print()

# 3. /u/ のF2に二峰性があるか。dip検定はラベル分割と無関係な問い
#    （データ全体の分布の形を見るだけ）なので、訓練・テスト分割は不要。
u_f2 <- pb$f2[pb$vowel == "u"]
cat("\n/u/ のF2: n =", length(u_f2), " dip検定 p =",
    round(dip.test(u_f2)$p.value, 4), "\n")
u_f2z <- pb$f2z[pb$vowel == "u"]
cat("/u/ の正規化F2:        dip検定 p =", round(dip.test(u_f2z)$p.value, 4), "\n")

# --- 実行結果 ---
# talker_raw_mean  : 54.0   talker_raw_sd  : 4.6
# talker_norm_mean : 39.5   talker_norm_sd : 8.3
# ii_mean          : 99.4   ii_sd          : 0.7
# vowel10_mean     : 91.4
#
# /u/ のF2: n = 152  dip検定 p = 0.8386
# /u/ の正規化F2:        dip検定 p = 0.9241
#
# 【出力の読み方】
# ・1問目は予想どおりの向きになった。正規化すると話者タイプの判別精度は
#   平均54.0%→39.5%へ下がる——消した情報（話者差）を当てさせようとすれば
#   精度が落ちる、という予想が確認された。**分類精度は必ずチャンスレベル
#   （最頻クラス、約43%）と比べる。** 3クラスだから33%が基準、ではない。
# ・ここから重要な原則が出てくる。**前処理は「良い前処理」「悪い前処理」に
#   分かれるのではなく、問いによって適否が変わる。** 母音を当てたいなら
#   正規化すべきで、話者の属性を当てたいなら正規化してはいけない。
#   ★話者認識や法音声学では、まさにこの理由で正規化を行わない。
# ・2問目: /i/ vs /I/ の2クラス判別は平均99.4%（SD 0.7pt）で、10母音の
#   91.4%よりはっきり高い。**分類の難しさはクラス数とクラス間の距離で
#   決まる。**「正解率が高い」という報告は、何クラスの課題かを書かなければ
#   意味を持たない。
# ・3問目: /u/ のF2は生でも正規化後でも p が0.84、0.92 と大きく、
#   **二峰性の証拠は無い**。/u/ は後舌母音で、英語では前寄りに発音する話者と
#   そうでない話者がいる（u-fronting）ため二峰になる可能性を考えたが、
#   1952年のこのデータでは分かれていない。
# ・★この「見つからなかった」という結果も、きちんとした答えである。
#   dip検定のp値が大きいことは「単峰だと証明された」ことではなく、
#   「このデータからは二峰の証拠が得られなかった」ことを意味する。
#   152トークンという規模で検出できる程度の二峰性は無かった、と読む。
